In [1]:
import sys
sys.path.append("..")

In [2]:
import os, json, glob
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow import keras
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from src.preprocess import preprocess_image

model = keras.models.load_model("../data/cnn_final.keras")

with open("../data/idx_to_letter.json") as f:
    idx_to_letter = {int(k): v for k, v in json.load(f).items()}
letter_to_idx = {v: k for k, v in idx_to_letter.items()}
letters = [idx_to_letter[i] for i in range(24)]

rows, X = [], []
for path in glob.glob("../data/real/*/*/*.jpg"):
    parts = path.replace("\\", "/").split("/")
    person, letter = parts[-3], parts[-2]
    if letter not in letter_to_idx:
        continue
    img = cv2.imread(path)
    if img is None:
        continue
    X.append(preprocess_image(img))
    rows.append({"person": person, "letter": letter,
                 "y": letter_to_idx[letter], "path": path})

X = np.array(X)
df = pd.DataFrame(rows)
print(len(df), "images chargées")
print(df.groupby("person").size())

3 images chargées
person
malak    3
dtype: int64


In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for ax, i in zip(axes.ravel(), np.random.default_rng(0).choice(len(X), 12, replace=False)):
    ax.imshow(X[i].squeeze(), cmap="gray")
    ax.set_title(df.loc[i, "letter"]); ax.axis("off")
plt.show()

In [ ]:
proba = model.predict(X, verbose=0)
df["pred"] = proba.argmax(axis=1)
df["conf"] = proba.max(axis=1)
df["ok"] = df["pred"] == df["y"]

print(f"Accuracy sur photos réelles : {df['ok'].mean():.3f}")
print(f"F1 macro : {f1_score(df['y'], df['pred'], average='macro'):.3f}")

print("\nAccuracy par personne :")
print(df.groupby("person")["ok"].mean().round(3))

print("\nAccuracy par lettre (les plus faibles) :")
print(df.assign(letter=df["letter"]).groupby("letter")["ok"].mean().sort_values().head(8).round(3))

In [ ]:
print(df.groupby("ok")["conf"].describe().round(3))

In [ ]:
cm = confusion_matrix(df["y"], df["pred"], labels=range(24))
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=letters, yticklabels=letters)
plt.xlabel("Prédit"); plt.ylabel("Réel"); plt.title("Photos réelles")
plt.show()

wrong = df[~df["ok"]].head(12)
fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for ax, (i, r) in zip(axes.ravel(), wrong.iterrows()):
    ax.imshow(X[i].squeeze(), cmap="gray")
    ax.set_title(f"réel {idx_to_letter[r['y']]} / prédit {idx_to_letter[r['pred']]}\nconf {r['conf']:.2f}", fontsize=9)
    ax.axis("off")
plt.show()

In [ ]:
real_results = {
    "n_images": int(len(df)),
    "n_persons": int(df["person"].nunique()),
    "accuracy": float(df["ok"].mean()),
    "f1_macro": float(f1_score(df["y"], df["pred"], average="macro")),
    "accuracy_by_person": df.groupby("person")["ok"].mean().round(4).to_dict(),
}
with open("../docs/real_results.json", "w") as f:
    json.dump(real_results, f, indent=2)